# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.25 — FAST
## Cubic-triad constraints, DOF, auxiliary elimination and isotropic-limit audit

`.3.3.24` a introduit :
\[
C_{\mu\nu\rho\sigma}=\sum_A e^{(A)}_\mu e^{(A)}_\nu e^{(A)}_\rho e^{(A)}_\sigma,
\qquad
\mathcal O_C=C^{\mu\nu\rho\sigma}\sigma_{\mu\nu}\sigma_{\rho\sigma}.
\]

Cette étape audite uniquement l'action candidate actuelle, sans ajouter silencieusement de cinétique au trièdre.

In [1]:
from __future__ import annotations
import sympy as sp, json, sys
from pathlib import Path

UPSTREAM = {
 "sha256":"7286f9ba6558a11b6160b4e6059352150572f636aac239578e51f5efd543f448",
 "size_bytes":26259,
 "candidate_found":True,
 "distinct_physics_validated":False,
 "triad_dynamics_materialized":False,
 "triad_constraint_algebra_audited":False,
 "triad_dof_classified":False,
 "real_data_novelty_authorized":False,
 "next_authorized":"AUDIT-CUBIC-TRIAD-CONSTRAINTS-DOF-AND-ISOTROPIC-LIMIT"
}
UPSTREAM_GATE = all([
 UPSTREAM["candidate_found"],
 not UPSTREAM["distinct_physics_validated"],
 not UPSTREAM["triad_dynamics_materialized"],
 not UPSTREAM["triad_constraint_algebra_audited"],
 not UPSTREAM["triad_dof_classified"],
 not UPSTREAM["real_data_novelty_authorized"],
])
assert UPSTREAM_GATE
print("UPSTREAM_GATE =",UPSTREAM_GATE)
print("P3324_CANONICAL_SHA256 =",UPSTREAM["sha256"])

UPSTREAM_GATE = True
P3324_CANONICAL_SHA256 = 7286f9ba6558a11b6160b4e6059352150572f636aac239578e51f5efd543f448


## 1 — Contraintes géométriques du trièdre

Trois vecteurs 4D \(e_{(A)}^\mu\) donnent 12 composantes.

Contraintes :
\[
u_\mu e_{(A)}^\mu=0 \quad (3),
\]
\[
g_{\mu\nu}e_{(A)}^\mu e_{(B)}^\nu=\delta_{AB}\quad (6).
\]

Donc on attend \(12-9=3\) paramètres d'orientation.

In [2]:
x=sp.symbols("x0:12", real=True)
E=sp.Matrix(3,4,x)
eta=sp.diag(-1,1,1,1)
u=sp.Matrix([1,0,0,0]); u_cov=eta*u

cons=[]
for A in range(3):
    eA=E.row(A).T
    cons.append((u_cov.T*eA)[0])
for A in range(3):
    for B in range(A,3):
        eA=E.row(A).T; eB=E.row(B).T
        cons.append((eA.T*eta*eB)[0]-(1 if A==B else 0))

J=sp.Matrix(cons).jacobian(x)
rows=[[0,1,0,0],[0,0,1,0],[0,0,0,1]]
subs={}
for A in range(3):
    for m in range(4):
        subs[E[A,m]]=rows[A][m]
rank=J.subs(subs).rank()

TRIAD_CONSTRAINT_JACOBIAN_RANK=rank
TRIAD_REDUCED_ORIENTATION_CONFIGURATION_COMPONENTS=12-rank
TRIAD_CONSTRAINTS_INDEPENDENT_PASS=(rank==9)

assert TRIAD_CONSTRAINTS_INDEPENDENT_PASS
assert TRIAD_REDUCED_ORIENTATION_CONFIGURATION_COMPONENTS==3
print("constraint_count =",len(cons))
print("rank =",rank)
print("orientation_components =",TRIAD_REDUCED_ORIENTATION_CONFIGURATION_COMPONENTS)
print("TRIAD_CONSTRAINTS_INDEPENDENT_PASS =",TRIAD_CONSTRAINTS_INDEPENDENT_PASS)

constraint_count = 9
rank = 9
orientation_components = 3
TRIAD_CONSTRAINTS_INDEPENDENT_PASS = True


## 2 — Pas de terme cinétique du trièdre

Dans `.3.3.24`,
\[
S_{\rm cand}=S_{\rm EA}+\frac{\zeta_C}{2}\int\sqrt{-g}\,\mathcal O_C.
\]

Aucun terme \(\nabla e_{(A)}\) n'est présent.

Après résolution des contraintes géométriques, les trois orientations \(q^a\) n'ont donc aucune vitesse \(\dot q^a\) dans le Lagrangien.

In [3]:
qd1,qd2,qd3=sp.symbols("qd1 qd2 qd3", real=True)
W=sp.hessian(sp.Integer(0),(qd1,qd2,qd3))
TRIAD_VELOCITY_HESSIAN_RANK=W.rank()
TRIAD_HAS_OWN_KINETIC_TERM=False
TRIAD_PRIMARY_MOMENTA_CONSTRAINT_COUNT=3
assert TRIAD_VELOCITY_HESSIAN_RANK==0
print("TRIAD_VELOCITY_HESSIAN_RANK =",TRIAD_VELOCITY_HESSIAN_RANK)
print("TRIAD_HAS_OWN_KINETIC_TERM =",TRIAD_HAS_OWN_KINETIC_TERM)
print("TRIAD_PRIMARY_MOMENTA_CONSTRAINT_COUNT =",TRIAD_PRIMARY_MOMENTA_CONSTRAINT_COUNT)

TRIAD_VELOCITY_HESSIAN_RANK = 0
TRIAD_HAS_OWN_KINETIC_TERM = False
TRIAD_PRIMARY_MOMENTA_CONSTRAINT_COUNT = 3


## 3 — Hessien d'orientation sur la branche alignée

Prenons :
\[
\Sigma=\mathrm{diag}(\lambda_1,\lambda_2,\lambda_3),
\qquad
\lambda_1+\lambda_2+\lambda_3=0.
\]

Pour \(R\in SO(3)\),
\[
\mathcal O_C(R)=\sum_i[(R^T\Sigma R)_{ii}]^2.
\]

On calcule le Hessien exact autour de \(R=I\).

In [4]:
l1,l2=sp.symbols("l1 l2", real=True)
l3=-l1-l2
a,b,c=sp.symbols("a b c", real=True)
Sigma=sp.diag(l1,l2,l3)

R12=sp.Matrix([[sp.cos(a),-sp.sin(a),0],[sp.sin(a),sp.cos(a),0],[0,0,1]])
R13=sp.Matrix([[sp.cos(b),0,-sp.sin(b)],[0,1,0],[sp.sin(b),0,sp.cos(b)]])
R23=sp.Matrix([[1,0,0],[0,sp.cos(c),-sp.sin(c)],[0,sp.sin(c),sp.cos(c)]])
R=sp.simplify(R12*R13*R23)
S=sp.simplify(R.T*Sigma*R)
OC=sp.simplify(sum(S[i,i]**2 for i in range(3)))

grad0=sp.Matrix([sp.diff(OC,v) for v in (a,b,c)]).subs({a:0,b:0,c:0})
H=sp.hessian(OC,(a,b,c)).subs({a:0,b:0,c:0}).applyfunc(sp.simplify)

expected=sp.diag(-4*(l1-l2)**2,-4*(l1-l3)**2,-4*(l2-l3)**2)
ORIENTATION_EIGENFRAME_STATIONARY_PASS=all(sp.simplify(v)==0 for v in grad0)
ORIENTATION_HESSIAN_EXACT_PASS=(sp.simplify(H-expected)==sp.zeros(3))
ORIENTATION_HESSIAN_DET=sp.factor(H.det())

assert ORIENTATION_EIGENFRAME_STATIONARY_PASS
assert ORIENTATION_HESSIAN_EXACT_PASS
print("gradient =",list(grad0))
print("Hessian =")
sp.pprint(H)
print("det(H) =",ORIENTATION_HESSIAN_DET)
print("ORIENTATION_EIGENFRAME_STATIONARY_PASS =",ORIENTATION_EIGENFRAME_STATIONARY_PASS)
print("ORIENTATION_HESSIAN_EXACT_PASS =",ORIENTATION_HESSIAN_EXACT_PASS)

gradient = [0, 0, 0]
Hessian =
⎡            2                                  ⎤
⎢-4⋅(l₁ - l₂)          0                0       ⎥
⎢                                               ⎥
⎢                             2                 ⎥
⎢      0        -4⋅(2⋅l₁ + l₂)          0       ⎥
⎢                                               ⎥
⎢                                              2⎥
⎣      0               0         -4⋅(l₁ + 2⋅l₂) ⎦
det(H) = -64*(l1 - l2)**2*(l1 + 2*l2)**2*(2*l1 + l2)**2
ORIENTATION_EIGENFRAME_STATIONARY_PASS = True
ORIENTATION_HESSIAN_EXACT_PASS = True


Pour des valeurs propres distinctes, le Hessien a rang 3.

Donc, pour \(\zeta_C\neq0\), les 3 contraintes primaires \(p_a\approx0\) et les 3 secondaires \(\chi_a=\partial V/\partial q^a\approx0\) forment génériquement trois paires de seconde classe.

In [5]:
triad_phase_dim=6
generic_first_class=0
generic_second_class=6
generic_dof=sp.Rational(triad_phase_dim-2*generic_first_class-generic_second_class,2)

isotropic_first_class=3
isotropic_second_class=0
isotropic_dof=sp.Rational(triad_phase_dim-2*isotropic_first_class-isotropic_second_class,2)

GENERIC_AUXILIARY_TRIAD_CONFIG_DOF=generic_dof
ISOTROPIC_DECOUPLED_TRIAD_CONFIG_DOF=isotropic_dof
AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS=(generic_dof==0 and isotropic_dof==0)

assert AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS
print("generic_dof =",generic_dof)
print("isotropic_dof =",isotropic_dof)
print("AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS =",AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS)

generic_dof = 0
isotropic_dof = 0
AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS = True


## 4 — Limite isotrope

\[
\zeta_C\to0\Rightarrow S_{\rm cand}\to S_{\rm EA}.
\]

Le nombre de DOF du trièdre reste \(0\to0\), mais la classe des contraintes change :
- \(\zeta_C\neq0\) : 3 paires seconde classe ;
- \(\zeta_C=0\) : 3 générateurs première classe d'une orientation découplée.

La limite est donc correcte au niveau de l'action et du nombre de DOF, mais singulière au niveau du rang/classification Dirac.

In [6]:
ISOTROPIC_ACTION_LIMIT_PASS=True
ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS=(generic_dof==isotropic_dof==0)
ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS=False
ISOTROPIC_CONSTRAINT_RANK_CONTINUOUS=False

assert ISOTROPIC_ACTION_LIMIT_PASS
assert ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS
assert not ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS

print("ISOTROPIC_ACTION_LIMIT_PASS =",ISOTROPIC_ACTION_LIMIT_PASS)
print("ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS =",ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS)
print("ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS =",ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS)
print("ISOTROPIC_CONSTRAINT_RANK_CONTINUOUS =",ISOTROPIC_CONSTRAINT_RANK_CONTINUOUS)

ISOTROPIC_ACTION_LIMIT_PASS = True
ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS = True
ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS = False
ISOTROPIC_CONSTRAINT_RANK_CONTINUOUS = False


## 5 — Élimination on-shell sur la branche alignée

Sur \(R=I\),
\[
\mathcal O_C=\lambda_1^2+\lambda_2^2+\lambda_3^2=\sigma^2.
\]

Le secteur cisaillement devient :
\[
-c_{13}\sigma^2+\frac{\zeta_C}{2}\sigma^2
=
-\left(c_{13}-\frac{\zeta_C}{2}\right)\sigma^2.
\]

Donc :
\[
\boxed{c_{13}^{\rm eff}=c_{13}-\zeta_C/2}.
\]

In [7]:
zeta,c13,sigma2=sp.symbols("zeta c13 sigma2", real=True)
Ltot=-c13*sigma2+zeta*sp.Rational(1,2)*sigma2
c13_eff=sp.simplify(c13-zeta/2)
ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS=sp.simplify(Ltot-(-c13_eff*sigma2))==0
assert ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS
print("c13_eff =",c13_eff)
print("ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS =",ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS)

c13_eff = c13 - zeta/2
ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS = True


## 6 — Branche stationnaire distincte : témoin zero-diagonal

Pour :
\[
\Sigma=\mathrm{diag}(s,-s,0),
\]
une rotation de \(45^\circ\) donne un cisaillement purement hors-diagonale dans le trièdre :
\[
\mathcal O_C=0.
\]

Cette orientation est aussi stationnaire au premier ordre.

Donc l'action auxiliaire ne sélectionne pas une branche cubique unique.

In [8]:
s=sp.symbols("s", real=True, nonzero=True)
Sigma0=sp.diag(s,-s,0)
R45=sp.Matrix([[sp.sqrt(2)/2,-sp.sqrt(2)/2,0],
               [sp.sqrt(2)/2, sp.sqrt(2)/2,0],
               [0,0,1]])
S45=sp.simplify(R45.T*Sigma0*R45)
diag45=[sp.simplify(S45[i,i]) for i in range(3)]
OC45=sp.simplify(sum(v**2 for v in diag45))

G12=sp.Matrix([[0,-1,0],[1,0,0],[0,0,0]])
G13=sp.Matrix([[0,0,-1],[0,0,0],[1,0,0]])
G23=sp.Matrix([[0,0,0],[0,0,-1],[0,1,0]])
first=[]
for G in (G12,G13,G23):
    dS=sp.simplify(S45*G-G*S45)
    first.append(sp.simplify(2*sum(S45[i,i]*dS[i,i] for i in range(3))))

ZERO_DIAGONAL_WITNESS_OC_ZERO_PASS=(OC45==0)
ZERO_DIAGONAL_WITNESS_STATIONARY_PASS=all(v==0 for v in first)
assert ZERO_DIAGONAL_WITNESS_OC_ZERO_PASS
assert ZERO_DIAGONAL_WITNESS_STATIONARY_PASS

print("S45 =")
sp.pprint(S45)
print("diag =",diag45)
print("O_C =",OC45)
print("first variations =",first)
print("ZERO_DIAGONAL_WITNESS_OC_ZERO_PASS =",ZERO_DIAGONAL_WITNESS_OC_ZERO_PASS)
print("ZERO_DIAGONAL_WITNESS_STATIONARY_PASS =",ZERO_DIAGONAL_WITNESS_STATIONARY_PASS)

S45 =
⎡0   -s  0⎤
⎢         ⎥
⎢-s  0   0⎥
⎢         ⎥
⎣0   0   0⎦
diag = [0, 0, 0]
O_C = 0
first variations = [0, 0, 0]
ZERO_DIAGONAL_WITNESS_OC_ZERO_PASS = True
ZERO_DIAGONAL_WITNESS_STATIONARY_PASS = True


## 7 — Classification scientifique

Le tenseur cubique reste une structure off-shell non isotrope.

Mais, dans l'action candidate actuelle :
- le trièdre est auxiliaire ;
- il ajoute 0 DOF propagatif ;
- la branche alignée se réduit à un déplacement de \(c_{13}\) ;
- plusieurs branches stationnaires existent ;
- aucun principe de sélection de branche n'est matérialisé.

Donc la distinctness on-shell n'est pas établie.

In [9]:
CUBIC_TRIAD_CONSTRAINT_ALGEBRA_AUDITED=True
CUBIC_TRIAD_DOF_CLASSIFIED=True
CUBIC_TRIAD_ORIENTATION_COMPONENTS=3
CUBIC_TRIAD_PROPAGATING_DOF_IN_CURRENT_ACTION=0
CUBIC_TRIAD_ALIGNED_BRANCH_EA_REDUNDANT=True
CUBIC_TRIAD_MULTIPLE_STATIONARY_BRANCHES_EXIST=True
CUBIC_TRIAD_BRANCH_SELECTION_PRINCIPLE_MATERIALIZED=False
CUBIC_TRIAD_OFFSHELL_STRUCTURAL_DISTINCTNESS_RETAINED=True
CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED=False

DYNAMIC_CUBIC_TRIAD_ACTION_MATERIALIZED=False
EMERGENT_3D_TO_4D_TESTED_HERE=False
GVH_SPECIFIC_MATTER_COUPLING_TESTED_HERE=False

GVH_DISTINCT_PHYSICS_VALIDATED=False
REAL_DATA_GVH_NOVELTY_INFERENCE_AUTHORIZED=False
KERR_GVH_NOVELTY_BENCHMARK_AUTHORIZED=False
QUANTIZATION_AS_DISTINCT_GVH_THEORY_READY=False

assert CUBIC_TRIAD_CONSTRAINT_ALGEBRA_AUDITED
assert CUBIC_TRIAD_DOF_CLASSIFIED
assert CUBIC_TRIAD_PROPAGATING_DOF_IN_CURRENT_ACTION==0
assert CUBIC_TRIAD_ALIGNED_BRANCH_EA_REDUNDANT
assert not CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED

print("CUBIC_TRIAD_CONSTRAINT_ALGEBRA_AUDITED =",CUBIC_TRIAD_CONSTRAINT_ALGEBRA_AUDITED)
print("CUBIC_TRIAD_DOF_CLASSIFIED =",CUBIC_TRIAD_DOF_CLASSIFIED)
print("CUBIC_TRIAD_PROPAGATING_DOF_IN_CURRENT_ACTION =",CUBIC_TRIAD_PROPAGATING_DOF_IN_CURRENT_ACTION)
print("CUBIC_TRIAD_ALIGNED_BRANCH_EA_REDUNDANT =",CUBIC_TRIAD_ALIGNED_BRANCH_EA_REDUNDANT)
print("CUBIC_TRIAD_MULTIPLE_STATIONARY_BRANCHES_EXIST =",CUBIC_TRIAD_MULTIPLE_STATIONARY_BRANCHES_EXIST)
print("CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED =",CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED)

CUBIC_TRIAD_CONSTRAINT_ALGEBRA_AUDITED = True
CUBIC_TRIAD_DOF_CLASSIFIED = True
CUBIC_TRIAD_PROPAGATING_DOF_IN_CURRENT_ACTION = 0
CUBIC_TRIAD_ALIGNED_BRANCH_EA_REDUNDANT = True
CUBIC_TRIAD_MULTIPLE_STATIONARY_BRANCHES_EXIST = True
CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED = False


In [10]:
G25_AUDIT_PASS=all([
 UPSTREAM_GATE,
 TRIAD_CONSTRAINTS_INDEPENDENT_PASS,
 TRIAD_REDUCED_ORIENTATION_CONFIGURATION_COMPONENTS==3,
 TRIAD_VELOCITY_HESSIAN_RANK==0,
 ORIENTATION_EIGENFRAME_STATIONARY_PASS,
 ORIENTATION_HESSIAN_EXACT_PASS,
 AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS,
 ISOTROPIC_ACTION_LIMIT_PASS,
 ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS,
 not ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS,
 ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS,
 ZERO_DIAGONAL_WITNESS_OC_ZERO_PASS,
 ZERO_DIAGONAL_WITNESS_STATIONARY_PASS,
 CUBIC_TRIAD_CONSTRAINT_ALGEBRA_AUDITED,
 CUBIC_TRIAD_DOF_CLASSIFIED,
 not CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED,
 not GVH_DISTINCT_PHYSICS_VALIDATED,
 not REAL_DATA_GVH_NOVELTY_INFERENCE_AUTHORIZED,
])

G25_OBSTRUCTIONS=[
 "CURRENT_CUBIC_TRIAD_HAS_NO_KINETIC_TERM",
 "CURRENT_CUBIC_TRIAD_HAS_ZERO_NEW_PROPAGATING_DOF",
 "GENERIC_ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT",
 "MULTIPLE_AUXILIARY_STATIONARY_BRANCHES_EXIST",
 "NO_NONARBITRARY_BRANCH_SELECTION_PRINCIPLE",
 "ISOTROPIC_LIMIT_CHANGES_DIRAC_CONSTRAINT_CLASS",
 "DYNAMIC_CUBIC_TRIAD_ACTION_NOT_MATERIALIZED",
 "NONREDUNDANCY_VS_GENERAL_LORENTZ_VIOLATING_EFT_NOT_PROVEN",
 "EMERGENT_3D_TO_4D_TRACK_NOT_TESTED_HERE",
 "OBSERVATIONAL_VIABILITY_NOT_AUTHORIZED",
]

G25_NEXT_AUTHORIZED="FORMALIZE-DYNAMIC-CUBIC-TRIAD-OR-FUNDAMENTAL-3D-ORIGIN-AND-RERUN-CONSTRAINTS-DOF"
assert G25_AUDIT_PASS

print("G25_AUDIT_PASS =",G25_AUDIT_PASS)
print("GVH_DISTINCT_PHYSICS_VALIDATED =",GVH_DISTINCT_PHYSICS_VALIDATED)
print("REAL_DATA_GVH_NOVELTY_INFERENCE_AUTHORIZED =",REAL_DATA_GVH_NOVELTY_INFERENCE_AUTHORIZED)
print("G25_OBSTRUCTIONS =",G25_OBSTRUCTIONS)
print("G25_NEXT_AUTHORIZED =",G25_NEXT_AUTHORIZED)

G25_AUDIT_PASS = True
GVH_DISTINCT_PHYSICS_VALIDATED = False
REAL_DATA_GVH_NOVELTY_INFERENCE_AUTHORIZED = False
G25_OBSTRUCTIONS = ['CURRENT_CUBIC_TRIAD_HAS_NO_KINETIC_TERM', 'CURRENT_CUBIC_TRIAD_HAS_ZERO_NEW_PROPAGATING_DOF', 'GENERIC_ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT', 'MULTIPLE_AUXILIARY_STATIONARY_BRANCHES_EXIST', 'NO_NONARBITRARY_BRANCH_SELECTION_PRINCIPLE', 'ISOTROPIC_LIMIT_CHANGES_DIRAC_CONSTRAINT_CLASS', 'DYNAMIC_CUBIC_TRIAD_ACTION_NOT_MATERIALIZED', 'NONREDUNDANCY_VS_GENERAL_LORENTZ_VIOLATING_EFT_NOT_PROVEN', 'EMERGENT_3D_TO_4D_TRACK_NOT_TESTED_HERE', 'OBSERVATIONAL_VIABILITY_NOT_AUTHORIZED']
G25_NEXT_AUTHORIZED = FORMALIZE-DYNAMIC-CUBIC-TRIAD-OR-FUNDAMENTAL-3D-ORIGIN-AND-RERUN-CONSTRAINTS-DOF


## Verdict

\[
\boxed{N_{\rm DOF}^{\rm triad}=0}
\]
pour l'action auxiliaire actuelle.

La branche alignée redonne un simple déplacement de \(c_{13}\), donc reste EA-redondante.

Le tenseur cubique demeure un **NOV-CAND structurel off-shell**, mais :
\[
\boxed{\texttt{GVH\_DISTINCT\_PHYSICS\_VALIDATED=False}}.
\]

La prochaine étape autorisée doit choisir entre :
1. une dynamique propre du trièdre cubique ;
2. une origine fondamentale 3D du trièdre ;

puis refaire contraintes, DOF et limite relativiste.

In [11]:
artifact={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.25_Cubic_Triad_Constraints_DOF_Auxiliary_Elimination_and_Isotropic_Limit_Audit_FAST",
 "upstream":UPSTREAM,
 "geometric_constraints":{
   "raw_components":12,
   "constraint_jacobian_rank":TRIAD_CONSTRAINT_JACOBIAN_RANK,
   "orientation_components":TRIAD_REDUCED_ORIENTATION_CONFIGURATION_COMPONENTS,
   "pass":TRIAD_CONSTRAINTS_INDEPENDENT_PASS
 },
 "kinetic":{
   "has_own_kinetic_term":TRIAD_HAS_OWN_KINETIC_TERM,
   "velocity_hessian_rank":TRIAD_VELOCITY_HESSIAN_RANK,
   "primary_momenta_constraints":TRIAD_PRIMARY_MOMENTA_CONSTRAINT_COUNT
 },
 "dof":{
   "generic_auxiliary_config_dof":str(GENERIC_AUXILIARY_TRIAD_CONFIG_DOF),
   "isotropic_decoupled_config_dof":str(ISOTROPIC_DECOUPLED_TRIAD_CONFIG_DOF),
   "no_new_propagating_dof_pass":AUXILIARY_TRIAD_NO_NEW_PROPAGATING_DOF_PASS
 },
 "onshell":{
   "aligned_branch_reduces_to_c13_shift":ALIGNED_BRANCH_REDUCES_TO_C13_SHIFT_PASS,
   "c13_eff":str(c13_eff),
   "zero_diagonal_stationary_witness":ZERO_DIAGONAL_WITNESS_STATIONARY_PASS,
   "multiple_stationary_branches":CUBIC_TRIAD_MULTIPLE_STATIONARY_BRANCHES_EXIST
 },
 "isotropic_limit":{
   "action_limit_pass":ISOTROPIC_ACTION_LIMIT_PASS,
   "physical_dof_count_continuous":ISOTROPIC_TRIAD_PHYSICAL_DOF_COUNT_CONTINUOUS_PASS,
   "constraint_class_continuous":ISOTROPIC_CONSTRAINT_CLASS_CONTINUOUS,
   "constraint_rank_continuous":ISOTROPIC_CONSTRAINT_RANK_CONTINUOUS
 },
 "distinctness":{
   "offshell_structural_distinctness_retained":CUBIC_TRIAD_OFFSHELL_STRUCTURAL_DISTINCTNESS_RETAINED,
   "onshell_physical_distinctness_established":CUBIC_TRIAD_ONSHELL_PHYSICAL_DISTINCTNESS_ESTABLISHED,
   "GVH_distinct_physics_validated":GVH_DISTINCT_PHYSICS_VALIDATED,
   "dynamic_cubic_triad_action_materialized":DYNAMIC_CUBIC_TRIAD_ACTION_MATERIALIZED,
   "emergent_3D_to_4D_tested_here":EMERGENT_3D_TO_4D_TESTED_HERE
 },
 "locks":{
   "real_data_GVH_novelty_inference_authorized":REAL_DATA_GVH_NOVELTY_INFERENCE_AUTHORIZED,
   "Kerr_GVH_novelty_benchmark_authorized":KERR_GVH_NOVELTY_BENCHMARK_AUTHORIZED,
   "quantization_as_distinct_GVH_theory_ready":QUANTIZATION_AS_DISTINCT_GVH_THEORY_READY
 },
 "verdict":{
   "G25_AUDIT_PASS":G25_AUDIT_PASS,
   "obstructions":G25_OBSTRUCTIONS,
   "next_authorized":G25_NEXT_AUTHORIZED
 }
}

export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.3.3.25_Cubic_Triad_Constraints_DOF_Auxiliary_Elimination_and_Isotropic_Limit_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact,indent=2,ensure_ascii=False),encoding="utf-8")
print("G25 artifact =",artifact_path)

G25 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.25_Cubic_Triad_Constraints_DOF_Auxiliary_Elimination_and_Isotropic_Limit_Audit_FAST.json


### Provenance

Classification recommandée :
`AI-F + C + NOV-CAND-AUDIT`

Pas de `NOV-PASS`.

Le résultat à conserver est que la non-isotropie off-shell de `.3.3.24` subsiste, mais l'action auxiliaire actuelle n'établit aucune nouvelle physique on-shell.